In [3]:
# ===========================
# COLAB — Stage-wise graphs PER DATASET for ALL benchmarks
# ✅ Now parses filenames by searching ONLY:
#   1) the word "Benchmark"
#   2) then the attack name (Scaling, Sybil, OnOff, Cumulative) anywhere after it
#
# Works with many filename variants.
# Outputs:
#   figs_by_dataset/<Attack>/<DatasetSlug>/<Attack>_<DatasetSlug>_F1_over_stage.png
#   figs_by_dataset/<Attack>/<DatasetSlug>/<Attack>_<DatasetSlug>_ASR_over_stage.png
# ===========================

import re
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files

# ---------------------------
# 0) Upload (ZIP or CSVs)
# ---------------------------
uploaded = files.upload()

for name in list(uploaded.keys()):
    if name.lower().endswith(".zip"):
        print(f"Extracting {name} ...")
        with zipfile.ZipFile(name, "r") as z:
            z.extractall("benchmark_data")

BASE_DIR = Path("benchmark_data") if Path("benchmark_data").exists() else Path(".")
print("BASE_DIR =", BASE_DIR.resolve())

raw_csvs = list(BASE_DIR.rglob("*.csv"))
print(f"Found {len(raw_csvs)} CSVs.")
if not raw_csvs:
    raise RuntimeError("No '*.csv' files found.")

# ---------------------------
# 1) Filename parsing: find "Benchmark" then attack name
# ---------------------------
ATTACKS = {
    "Scaling":   ["scaling", "ModelReplacement"],
    "Sybil":     ["sybil","Sybil"],
    "OnOff":     ["onoff", "on-off", "on_off", "OnOff"],
    "Cumulative":["cumulative", "slowdrift", "slow_drift", "slow-drift", "drift"],
}

def detect_attack_from_name(filename: str):
    s = filename.lower()
    i = s.find("benchmark")
    if i < 0:
        return None
    tail = s[i:]  # only search after "benchmark"
    for canonical, aliases in ATTACKS.items():
        for a in aliases:
            if a in tail:
                return canonical
    return None

def slugify(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return s[:140] if s else "UnknownDataset"

def dataset_from_name(filename: str, attack: str):
    """
    Extract dataset name as the chunk between 'Results' (if present) and '__Raw_Results'
    otherwise fallback to removing known tokens and using remaining filename.
    """
    name = filename

    # Common pattern: ...Results__<DATASET>__Raw_Results.csv
    m = re.search(r"Results__+(?P<ds>.+)__Raw_Results\.csv$", name, flags=re.IGNORECASE)
    if m:
        return m.group("ds").strip()

    # Alternative: Benchmark_<attack>__<DATASET>__Raw_Results.csv
    m = re.search(r"Benchmark.*?__+(?P<ds>.+)__Raw_Results\.csv$", name, flags=re.IGNORECASE)
    if m:
        return m.group("ds").strip()

    # Fallback: strip suffix/prefix and return what remains
    base = re.sub(r"__Raw_Results\.csv$", "", name, flags=re.IGNORECASE)
    base = re.sub(r"Benchmark", "", base, flags=re.IGNORECASE)
    base = re.sub(r"Results", "", base, flags=re.IGNORECASE)
    base = re.sub(r"Raw_Results", "", base, flags=re.IGNORECASE)
    # also strip attack tokens
    for a in ATTACKS.get(attack, []):
        base = re.sub(a, "", base, flags=re.IGNORECASE)
    base = base.strip("_- ")
    return base if base else "UnknownDataset"

# ---------------------------
# 2) Load CSVs and attach Attack + Dataset
# ---------------------------
REQUIRED = {"Seed", "Strategy", "Stage", "F1", "ASR", "Time"}

frames = []
skipped = []

for p in raw_csvs:
    attack = detect_attack_from_name(p.name)
    if attack is None:
        skipped.append((p, "attack not detected (needs 'Benchmark' + attack name in filename)"))
        continue

    ds = dataset_from_name(p.name, attack)

    df = pd.read_csv(p)
    missing = REQUIRED - set(df.columns)
    if missing:
        skipped.append((p, f"missing columns: {missing}"))
        continue

    df = df[list(REQUIRED)].copy()
    df["Attack"] = attack
    df["Dataset"] = ds
    frames.append(df)

print(f"Loaded {len(frames)} CSVs; skipped {len(skipped)}.")
if skipped:
    print("\nSkipped files (first 10):")
    for item in skipped[:10]:
        print(" -", item[0], "=>", item[1])

if not frames:
    raise RuntimeError("No valid CSVs loaded. Ensure filenames contain 'Benchmark' and the attack name.")

df_all = pd.concat(frames, ignore_index=True)

print("\nDetected attacks:", sorted(df_all["Attack"].unique()))
print("Detected datasets:", len(df_all["Dataset"].unique()))

# ---------------------------
# 3) Stage-wise mean PER DATASET (avg over seeds)
# ---------------------------
df_mean = (
    df_all.groupby(["Attack", "Dataset", "Strategy", "Stage"], as_index=False)[["F1", "ASR", "Time"]]
          .mean()
          .rename(columns={"F1":"F1_mean", "ASR":"ASR_mean", "Time":"Time_mean"})
)

# ---------------------------
# 4) Plot
# ---------------------------
def plot_stagewise(df_bd, attack, dataset, metric_col, ylabel, outpath: Path):
    df_bd = df_bd.sort_values(["Stage","Strategy"])
    fig, ax = plt.subplots(figsize=(10, 4.8), dpi=180)

    for strat, g in df_bd.groupby("Strategy"):
        ax.plot(g["Stage"].values, g[metric_col].values,
                marker="o", linewidth=1.8, markersize=3.0, label=strat)

    ax.set_title(f"{attack} — {dataset}\nMean {ylabel} over stages (avg. across seeds)", fontsize=12)
    ax.set_xlabel("Stage")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
    ax.set_xticks(sorted(df_bd["Stage"].unique()))

    if "ASR" in metric_col.upper():
        ax.set_ylim(-0.02, 1.02)
    if "F1" in metric_col.upper():
        ax.set_ylim(-0.02, 1.02)

    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=9)

    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)

# ---------------------------
# 5) Generate ALL attacks x ALL datasets
# ---------------------------
OUT_DIR = Path("figs_by_dataset")
generated = []

for attack in sorted(df_mean["Attack"].unique()):
    df_a = df_mean[df_mean["Attack"] == attack]
    for dataset in sorted(df_a["Dataset"].unique()):
        df_ad = df_a[df_a["Dataset"] == dataset]
        ds_slug = slugify(dataset)

        f1_out  = OUT_DIR / attack / ds_slug / f"{attack}_{ds_slug}_F1_over_stage.png"
        asr_out = OUT_DIR / attack / ds_slug / f"{attack}_{ds_slug}_ASR_over_stage.png"

        plot_stagewise(df_ad, attack, dataset, "F1_mean",  "F1-score", f1_out)
        plot_stagewise(df_ad, attack, dataset, "ASR_mean", "ASR",      asr_out)

        generated += [f1_out, asr_out]

print(f"\n✅ Generated {len(generated)} plots in: {OUT_DIR.resolve()}")

# ---------------------------
# 6) ZIP + download
# ---------------------------
zip_name = "figs_stagewise_by_dataset_ALL_attacks.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in generated:
        z.write(p, arcname=p.as_posix())

print(f"✅ Zipped: {zip_name}")
files.download(zip_name)

Saving Benchmark_Sybil_Results__Samsung_SNH_1011_N_Webcam_gafgyt_attacks__Raw_Results_Sybil.csv to Benchmark_Sybil_Results__Samsung_SNH_1011_N_Webcam_gafgyt_attacks__Raw_Results_Sybil (1).csv
Saving Benchmark_Sybil_Results__Provision_PT_838_Security_Camera_mirai_attacks__Raw_Results_Sybil.csv to Benchmark_Sybil_Results__Provision_PT_838_Security_Camera_mirai_attacks__Raw_Results_Sybil (1).csv
Saving Benchmark_Sybil_Results__Philips_B120N10_Baby_Monitor_mirai_attacks__Raw_Results_Sybil.csv to Benchmark_Sybil_Results__Philips_B120N10_Baby_Monitor_mirai_attacks__Raw_Results_Sybil (1).csv
Saving Benchmark_Sybil_Results__Philips_B120N10_Baby_Monitor_gafgyt_attacks__Raw_Results_Sybil.csv to Benchmark_Sybil_Results__Philips_B120N10_Baby_Monitor_gafgyt_attacks__Raw_Results_Sybil (1).csv
Saving Benchmark_Sybil_Results__Ennio_Doorbell_gafgyt_attacks__Raw_Results_Sybil.csv to Benchmark_Sybil_Results__Ennio_Doorbell_gafgyt_attacks__Raw_Results_Sybil (1).csv
Saving Benchmark_Sybil_Results__Ecobee_T

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>